In [ ]:
import requests
from bs4 import BeautifulSoup

html = requests.get('http://127.0.0.1:5500/homework/simple.html')
soup = BeautifulSoup(html.text, 'html.parser')

tbody = soup.find('tbody')
rows = tbody.find_all('tr')

weather_data = []
print("Weather data: ")
for tr in rows:
    tds = tr.find_all('td')
    day = tds[0].text
    temp = tds[1].text 
    condition = tds[2].text
    weather_data.append({'day': day, 'temp': temp, 'condition': condition})
    print(f'day: {day}, temperature: {temp}, condition: {condition}')
print("-"*60)
print('\n')
# print(weather_data)
highest_temp = weather_data[0].copy()
temps = []
for row in weather_data:
    temp_w = int(row['temp'][:2])
    temp_h = int(highest_temp['temp'][:2])
    temps.append(temp_w)
    if temp_w > temp_h: 
        highest_temp['day'] = row['day']
        highest_temp['temp'] = row['temp']
        highest_temp['condition'] = row['condition']
print(f"The highest temperature day: {highest_temp['day'], highest_temp['temp'], highest_temp['condition']}")

sunny_day = []
for row in weather_data:
    if row['condition'] == "Sunny":
        sunny_day.append(row)

print("Sunny days: ")
for day in sunny_day:
    print(f'day: {day['day']}, temp: {day['temp']}, condition: {day['condition']}')

average_temp = sum(temps)/len(temps)
print(f"The average temperature: {average_temp}")

Job title: Senior Python Developer, Companies: Payne, Roberts and Davis, Location: Stewartbury, AA, Description: https://www.realpython.com, Application link: https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html
------------------------------
Job title: Energy engineer, Companies: Vasquez-Davidson, Location: Christopherville, AA, Description: https://www.realpython.com, Application link: https://realpython.github.io/fake-jobs/jobs/energy-engineer-1.html
------------------------------
Job title: Legal executive, Companies: Jackson, Chambers and Levy, Location: Port Ericaburgh, AA, Description: https://www.realpython.com, Application link: https://realpython.github.io/fake-jobs/jobs/legal-executive-2.html
------------------------------
Job title: Fitness centre manager, Companies: Savage-Bradley, Location: East Seanview, AP, Description: https://www.realpython.com, Application link: https://realpython.github.io/fake-jobs/jobs/fitness-centre-manager-3.html
----------

In [ ]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import csv

class JobScraper:
    def __init__(self, db_name="jobs.db"):
        self.db_name = db_name
        self.create_table()

    def create_table(self):
        with sqlite3.connect(self.db_name) as conn:
            cursor = conn.cursor()
            query = """
                CREATE TABLE IF NOT EXISTS jobs(
                    Job_title TEXT,
                    Company_name TEXT,
                    Location TEXT,
                    Job_description TEXT,
                    Application_link TEXT
                )
            """
            cursor.execute(query)

    def scrape_jobs(self):
        data = []
        url = 'https://realpython.github.io/fake-jobs'
        html = requests.get(url)
        soup = BeautifulSoup(html.text, 'html.parser')

        card_contents = soup.findAll(class_='card')

        for card_content in card_contents:
            job = card_content.find('h2').text
            company = card_content.find('h3').text
            location = card_content.findAll(class_='location')[0].text.strip()
            description = card_content.findAll('a')[0]['href']
            apply = card_content.findAll('a')[1]['href']
            data.append({'Job_title': job, 'Company': company, 'Location': location, 'Description': description, 'Application_link': apply})
        return data

    def save_jobs(self, jobs_list):
        with sqlite3.connect('jobs.db') as conn:
            cursor = conn.cursor()

            for job in jobs_list:
                cursor.execute(
                    "SELECT Job_description, Application_link FROM jobs WHERE Job_title=? AND Company_name=? AND Location=?",
                    (job['Job_title'], job['Company'], job['Location'])
                )
                existing = cursor.fetchone()

                if existing is None:
                    cursor.execute(
                        "INSERT INTO jobs VALUES(?, ?, ?, ?, ?)", (
                            job['Job_title'], job['Company'], job['Location'], job['Description'], job['Application_link']
                        )
                    )
                else:
                    old_description, old_link = existing
                    if old_description != job['Description'] or old_link != job['Application_link']:
                        cursor.execute(
                            "UPDATE jobs SET Job_description=?, Application_link=? WHERE Job_title=? AND Company_name=? AND Location=?",
                            (job['Description'], job['Application_link'], job['Job_title'], job['Company'], job['Location'])
                        )


    def filter_jobs(self, location=None, company=None):
        with sqlite3.connect(self.db_name) as conn:
            cursor = conn.cursor()
            if location:
                cursor.execute("SELECT * FROM jobs WHERE Location=?", (location,))
            elif company:
                 cursor.execute("SELECT * FROM jobs WHERE Company_name=?", (company,))
            else: 
                 cursor.execute("SELECT * FROM jobs")
            job_list = cursor.fetchall()
            self.export_to_csv(job_list)

    def export_to_csv(self, jobs_list, filename="filtered_jobs.csv"):
        with open(filename, 'wt', newline='') as file:
            writer = csv.writer(file)
            columns = ['Job_title', 'Company_name','Location', 'Job_description', 'Application_link']
            writer.writerow(columns)
            for row in jobs_list:
                writer.writerow([row[0], row[1], row[2], row[3], row[4]])
    def run(self):
        jobs = self.scrape_jobs()
        self.save_jobs(jobs)
        


scraper = JobScraper()
scraper.run()
scraper.filter_jobs(company='Mitchell, Jones and Olson')

NameError: name 'data' is not defined

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import json

import time

chrome_options = Options()
# chrome_options.add_argument("--window-position=-2000,-100")  # Position window on the second screen
chrome_options.add_argument("--window-position=1020,0")

driver = webdriver.Chrome(chrome_options)

driver.get('https://www.demoblaze.com/')
xpath = """//*[@id="itemc"]"""

time.sleep(2)
element = driver.find_element(By.XPATH, "//a[@onclick=\"byCat(\'notebook\')\"]")
element.click()
time.sleep(3)

data = []
contents = driver.find_elements(By.CLASS_NAME, "card-block") 
for content in contents:
    name = content.find_element(By.TAG_NAME, 'a')
    price = content.find_element(By.TAG_NAME, 'h5')
    description = content.find_element(By.TAG_NAME, 'p')
    print(f"{name.text}\n {price.text}\n {description.text}")
    print("-"*30)
    data.append({'name': name.text, 'price': price.text, 'description': description.text})
time.sleep(5)
driver.quit()

with open('data.json', 'wt') as file:
    writer = json.dump(data, file, indent=2)
    print("Successfully saved")
